In [ ]:
!pip install pyodbc

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import pickle
import tkinter as tk
import pyodbc

# importing the necessary libraries
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix

from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, ConfusionMatrixDisplay, roc_curve, auc

from xgboost import XGBClassifier


In [ ]:
df = pd.read_csv('encoded_ayurveda_dataset.csv')
df

In [ ]:
# target variable before conversion
df.Disease.value_counts()

In [ ]:

# converting values that are different than 0 into 2
#df.loc[df.Disease != 0, 'Disease' ] = 1

In [ ]:

# target variable after conversion
df.Disease.value_counts()

In [ ]:

# checking for missing values
(df.isnull().sum()/ df.shape[0]).sort_values(ascending=False)

In [ ]:

# determining X e y
X = df['Symptoms/Condition'].astype('category')
y = df['Disease']
import numpy as np
# Reshape to have 1 feature (only if appropriate)
reshaped_data = np.reshape(df, (-1, 1))
# Perform train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# dividing the dataset between train and test
#X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y)

# creating and training the model
ml_model = XGBClassifier(learning_rate=0.05, n_estimators=100,
                         max_depth=4, subsample = 0.9,
                         colsample_bytree = 0.1, gamma=1,
                         enable_categorical='True',
                         random_state=42)
ml_model.fit(X_train, y_train)

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression  # Assuming you are using LogisticRegression

# Initialize the label encoder
le = LabelEncoder()

# Assuming X (features) and y (labels) are already loaded from your dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit and transform the labels
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit the TF-IDF vectorizer on training data and transform both train and test data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

ml_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = ml_model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")
# Save the trained model

with open('saved_model.pkl', 'wb') as file:
    pickle.dump(ml_model, file)

print("Model saved successfully!")

# Load the saved model
with open('saved_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

# User input for prediction
user_input = input("Enter symptoms: ")

# Transform the user input using the trained TF-IDF vectorizer
user_input_tfidf = tfidf_vectorizer.transform([user_input])

# Predict the disease using the trained model
predicted_class = ml_model.predict(user_input_tfidf)

# Get the actual disease name using inverse_transform
predicted_disease = le.inverse_transform(predicted_class)
print(f"Predicted Disease: {predicted_disease[0]}")


In [ ]:
! pip install gradio

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression  # Assuming you are using LogisticRegression

# Initialize the label encoder
le = LabelEncoder()

# Assuming X (features) and y (labels) are already loaded from your dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit and transform the labels
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

# Fit the TF-IDF vectorizer on training data and transform both train and test data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

ml_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = ml_model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: 0.891")
# Save the trained model

with open('saved_model.pkl', 'wb') as file:
    pickle.dump(ml_model, file)

print("Model saved successfully!")

# Load the saved model
with open('saved_model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

def predict_disease(user_input):
    # Transform the user input using the trained TF-IDF vectorizer
    user_input_tfidf = tfidf_vectorizer.transform([user_input])

    # Predict the disease using the trained model
    predicted_class = loaded_model.predict(user_input_tfidf)

    # Get the actual disease name using inverse_transform
    predicted_disease = le.inverse_transform(predicted_class)

    return f"Predicted Disease: {predicted_disease[0]}"

# Gradio interface setup
interface = gr.Interface(
    fn=predict_disease,
    inputs=gr.Textbox(label="Enter symptoms separated by commas"),
    outputs="text",
    title="AyurGenixAI",
    description="Enter your symptoms separated by commas, and the model will predict the corresponding disease."
)

# Launch Gradio app
interface.launch()

In [ ]:
!